# terrio quickstart

This notebook runs entirely in your browser, in [JupyterLite](https://jupyterlite.readthedocs.io): terrio is compiled to WebAssembly and nothing is installed or uploaded. Edits you save stay in this browser, so download the notebook to keep them elsewhere.

Browser builds of terrio do not open URLs, so this notebook downloads its data first. To read cloud-optimized GeoTIFFs in place with HTTP range requests, run the [full notebook](https://github.com/opengeos/terrio/blob/main/docs/notebooks/quickstart.ipynb) in Google Colab or locally.

## Install

The site carries terrio's Pyodide wheel. Installing takes a few seconds on the first run.

In [ ]:
%pip install -q terrio matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import terrio

terrio.__version__

## Download the data

In [ ]:
from pyodide.http import pyfetch


async def download(url, path):
    """Saves a URL to a file in the browser's memory."""
    response = await pyfetch(url)
    with open(path, "wb") as file:
        file.write(await response.bytes())


data = "https://raw.githubusercontent.com/opengeos/data/main"

## A digital elevation model

An SRTM elevation model of the Sierra Nevada (16 MB).

In [ ]:
await download(f"{data}/raster/srtm90.tif", "srtm90.tif")
dem = terrio.open_raster("srtm90.tif", lazy=True)
print(dem.width, dem.height, dem.dtype, dem.crs)
print("overviews:", dem.overviews)

The smallest overview is a quick look at the whole image:

In [ ]:
overview = dem.read(1, overview=len(dem.overviews) - 1)
plt.imshow(np.where(overview == dem.nodata, np.nan, overview), cmap="terrain")
plt.colorbar(label="elevation (m)")
plt.title("srtm90, smallest overview");

A full-resolution window, written to a GeoTIFF with its statistics:

In [ ]:
terrio.convert("srtm90.tif", "window.tif", srcwin=(1000, 500, 1024, 1024))
terrio.info("window.tif", stats=True)["bands"][0]["statistics"]

## Terrain analysis

`terrio.dem` matches `gdaldem` pixel for pixel.

In [ ]:
shade = terrio.dem("window.tif", "hillshade", multidirectional=True)
slope = terrio.dem("window.tif", "slope")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(shade.read(1), cmap="gray")
axes[0].set_title("hillshade")
im = axes[1].imshow(np.ma.masked_equal(slope.read(1), slope.nodata), cmap="magma")
axes[1].set_title("slope (degrees)")
fig.colorbar(im, ax=axes[1]);

## Reprojection

`terrio.warp` is `gdalwarp`: here to UTM zone 11N at 180 m with bilinear resampling.

In [ ]:
utm = terrio.warp("window.tif", "window_utm.tif", t_srs="EPSG:32611", tr=(180, 180), resampling="bilinear")
print(utm.width, utm.height, utm.crs)
print(utm.transform)

## A multispectral image and raster algebra

A Landsat 7 scene of San Francisco Bay (bands 1 to 3 are near infrared, red and green). `terrio.calc` is `gdal_calc.py`, with NumPy's typing rules.

In [ ]:
await download(f"{data}/raster/landsat7.tif", "landsat7.tif")
# The image declares 0 as nodata, but in 8 bits the red band is 0 over dense vegetation, so
# ignore the nodata value (hide_nodata, gdal_calc.py's --hideNoData). Water, 0 in both bands,
# divides 0 by 0 and is left as NaN.
ndvi = terrio.calc("(A - B * 1.0) / (A + B * 1.0)", A="landsat7.tif", A_band=1, B="landsat7.tif", B_band=2, type="Float32", hide_nodata=True)
values = ndvi.read(1)
plt.imshow(values, cmap="RdYlGn", vmin=-0.5, vmax=0.8)
plt.colorbar(label="NDVI")
plt.title("Landsat 7, NDVI");

## Vector data

The world's countries as GeoJSON.

In [ ]:
await download(f"{data}/world/countries.geojson", "countries.geojson")
countries = terrio.open_vector("countries.geojson")
layer = countries.layers[0]
print(layer.name, layer.geometry_type, len(layer), layer.crs)
print(layer.fields)

Filter with SQL, reproject, and write to other formats:

In [ ]:
terrio.convert("countries.geojson", "united.gpkg", where="name LIKE 'United%'", t_srs="EPSG:3857")
united = terrio.open_vector("united.gpkg").layers[0]
print(united.crs, [f["properties"]["name"] for f in united.to_geojson()["features"]])

In [ ]:
for ext in ["fgb", "parquet", "shp", "kml"]:
    terrio.write_vector(layer, f"countries.{ext}")
    back = terrio.open_vector(f"countries.{ext}").layers[0]
    print(f"countries.{ext}: {len(back)} features")

A quick plot of the polygons:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for feature in layer.to_geojson()["features"]:
    geometry = feature["geometry"]
    polygons = [geometry["coordinates"]] if geometry["type"] == "Polygon" else geometry["coordinates"]
    for polygon in polygons:
        ring = np.array(polygon[0])
        ax.fill(ring[:, 0], ring[:, 1], alpha=0.5, edgecolor="k", linewidth=0.3)
ax.set_aspect("equal")
ax.set_title("countries.geojson");

## Rasterize

In [ ]:
mask = terrio.rasterize(layer, burn=1, tr=(0.5, 0.5), ot="Byte", init=0)
plt.imshow(mask.read(1), cmap="Greys")
plt.title("countries rasterized at 0.5°");

## Coordinates

Transformations use the operations GDAL picks, for every EPSG CRS.

In [ ]:
xs, ys = terrio.transform_points([-119.5], [37.75], "EPSG:4326", "EPSG:32611")
print(xs[0], ys[0])

## Next

- [Documentation](https://terrio.dev)
- [Python API](https://terrio.dev/python/)
- [Try terrio's file converter in the browser](https://terrio.dev/playground/)